![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 04: Data Manipulation)**

**Session 4H: Case Study - Word Count Application**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 04.</td>
</tr>
<tr>
<td align="left">Estimated duration</td>
<td>Approximately 40 minutes, based on 240 minutes of M04 practical/self-learning work divided across six M04 notebooks.</td>
</tr>
<tr>
<td align="left">Main packages</td>
<td><code>pyspark</code> and Python standard-library path and package handling. Spark also requires a Java runtime.</td>
</tr>
<tr>
<td align="left">Data files</td>
<td><code>shakespeare.txt</code> from the public <code>Jupyter/data/</code> folder.</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

- [1. Overview and Learning Goals](#1-overview-and-learning-goals)
- [2. Setup and Data Files](#2-setup-and-data-files)
- [3. Creating Base RDDs and Pair RDDs](#3-creating-base-rdds-and-pair-rdds)
- [4. Counting With Pair RDDs](#4-counting-with-pair-rdds)
- [5. Unique Words and Mean Counts](#5-unique-words-and-mean-counts)
- [6. Word-Count Helpers](#6-word-count-helpers)
- [7. Shakespeare Word Count Case Study](#7-shakespeare-word-count-case-study)
- [8. Student Tasks](#8-student-tasks)
- [9. Practical Self-Checks](#9-practical-self-checks)
- [10. Reflection and References](#10-reflection-and-references)


<a id="1-overview-and-learning-goals"></a>

### 1. Overview and Learning Goals

This session builds a small Spark word-count application with pair RDDs. You will start from a short list of words, compare `groupByKey()` and `reduceByKey()`, write reusable word-count helpers, and then apply the workflow to the Complete Works of William Shakespeare text file from Project Gutenberg.

The practical is intentionally structured as a case study. Early cells are mostly worked examples. Later cells use starter code so that you complete the core word-count and text-cleaning steps yourself before running the larger text-file workflow.

By the end of this practical, you should be able to:

1. create base RDDs from Python collections;
2. use `map`, `flatMap`, `groupByKey`, `reduceByKey`, and `takeOrdered`;
3. explain why `reduceByKey()` is preferred over `groupByKey()` for scalable word counts;
4. write a reusable RDD word-count helper;
5. clean text consistently before counting words;
6. run visible formative checks to diagnose your notebook state.


<a id="2-setup-and-data-files"></a>

### 2. Setup and Data Files

This notebook uses PySpark and the public SIT742 `shakespeare.txt` text file. Choose the execution option that matches where you are running the notebook.

#### Option A: Google Colab / online execution

Use this option when you are running the notebook in Google Colab or another online notebook environment, or when you do not have the SIT742 repository cloned locally. The setup code downloads the required text file from the public SIT742 GitHub repository into the notebook runtime and installs PySpark if required. Spark also needs a Java runtime; the setup code checks for Java and uses the online runtime's package manager when available.

#### Option B: Local repository execution

Use this option only if you have cloned the SIT742 repository locally and are running the notebook from its original folder structure. The file paths below are relative to this notebook location. Local execution expects Java and PySpark to be available in your active Python environment.

Keep `EXECUTION_MODE = "online"` for Option A. Change it to `"local"` only for Option B.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import shutil
import subprocess
import sys
import tempfile

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.
PYSPARK_PACKAGE = "pyspark==3.5.1"
required_files = ["shakespeare.txt"]


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m04h_data_"))
    downloaded_paths = {}
    for filename in required_files:
        url = f"{PUBLIC_DATA_BASE_URL}/{filename}"
        local_file = data_dir / filename
        urlretrieve(url, local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([base / "Jupyter" / "data", base / "data", base / "SIT742" / "Jupyter" / "data"])

    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate, {filename: candidate / filename for filename in required_files}

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find the public SIT742 data files locally. "
        "Set EXECUTION_MODE = 'online' or run this notebook from a cloned SIT742 repository.\n"
        f"Checked:\n{checked}"
    )


def ensure_java_available():
    if shutil.which("java"):
        return
    if shutil.which("apt-get"):
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "openjdk-17-jdk-headless"], check=True)
        return
    raise RuntimeError(
        "Java is required for Spark. Use Google Colab, or install a supported Java runtime locally."
    )


def ensure_python_module(module_name, package_name):
    if importlib.util.find_spec(module_name) is not None:
        return
    if EXECUTION_MODE == "online":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return
    raise RuntimeError(
        f"{package_name} is required. Install it in the active environment with: python -m pip install {package_name}"
    )


if EXECUTION_MODE == "online":
    DATA_DIR, data_files = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR, data_files = find_local_data_dir(required_files)
else:
    raise ValueError("EXECUTION_MODE must be either 'online' or 'local'.")

ensure_java_available()
ensure_python_module("pyspark", PYSPARK_PACKAGE)

shakespeare_path = data_files["shakespeare.txt"]
DataSet = shakespeare_path
print("Execution mode:", EXECUTION_MODE)
print("Data directory:", DATA_DIR)
print("Shakespeare file:", shakespeare_path, shakespeare_path.stat().st_size, "bytes")


#### 2.1 Start Spark

Create a local Spark session and keep a reference to its SparkContext as `sc`. The examples below use the RDD API, so `sc` is the main object you will call.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SIT742-M04H-WordCount")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", sc.master)
print(type(sc))


<a id="3-creating-base-rdds-and-pair-rdds"></a>

### 3. Creating Base RDDs and Pair RDDs

In this part of the practical, you will create a base RDD with `parallelize`, transform it with `map`, and create pair RDD records of the form `(word, 1)`.

#### 3.1 Create a base RDD


In [ ]:
wordsList = ["cat", "elephant", "rat", "rat", "cat"]
wordsRDD = sc.parallelize(wordsList, 4)

print(type(wordsRDD))
print(wordsRDD.collect())


#### 3.2 Define a simple pluralisation function

This function adds `s` to each word. It is deliberately simple and does not attempt English pluralisation rules.


In [ ]:
def makePlural(word):
    return word + "s"

print(makePlural("cat"))


#### 3.3 Apply `makePlural` to the base RDD

`map` applies a function to every element in an RDD. `collect` brings the small result back to the notebook driver for display.


In [ ]:
pluralRDD = wordsRDD.map(makePlural)
print(pluralRDD.collect())


#### 3.4 Pass a `lambda` function to `map`

The same simple transformation can be written inline with a lambda function.


In [ ]:
pluralLambdaRDD = wordsRDD.map(lambda word: word + "s")
print(pluralLambdaRDD.collect())


#### 3.5 Length of each word

Use `map` to calculate the number of characters in each plural word.


In [ ]:
pluralLengths = pluralRDD.map(lambda word: len(word)).collect()
print(pluralLengths)


#### 3.6 Pair RDDs

A pair RDD stores key-value records. For word count, each word starts as `(word, 1)`.


In [ ]:
wordPairs = wordsRDD.map(lambda word: (word, 1))
print(wordPairs.collect())


<a id="4-counting-with-pair-rdds"></a>

### 4. Counting With Pair RDDs

There are multiple ways to count words with pair RDDs. `groupByKey()` is easy to understand, but `reduceByKey()` is usually the better distributed approach because it can combine values within partitions before moving data across the cluster.

#### 4.1 `groupByKey()` approach


In [ ]:
wordsGrouped = wordPairs.groupByKey()
for key, value in wordsGrouped.collect():
    print(f"{key}: {list(value)}")


#### 4.2 Use `groupByKey()` to obtain counts

The values returned by `groupByKey()` are iterators. You can sum each iterator to create a count per word.


In [ ]:
wordCountsGrouped = wordsGrouped.map(lambda item: (item[0], sum(item[1])))
print(sorted(wordCountsGrouped.collect()))


#### 4.3 Counting with `reduceByKey`

`reduceByKey()` combines values for each key with a two-argument function.


In [ ]:
wordCounts = wordPairs.reduceByKey(lambda a, b: a + b)
print(sorted(wordCounts.collect()))


#### 4.4 All together

The full short-list word count can be written as one chained expression.


In [ ]:
wordCountsCollected = (
    wordsRDD
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)
print(sorted(wordCountsCollected))


<a id="5-unique-words-and-mean-counts"></a>

### 5. Unique Words and Mean Counts

This section uses the small counted RDD to calculate two simple summary values.

#### 5.1 Unique words


In [ ]:
uniqueWords = len(wordCountsCollected)
print(uniqueWords)


#### 5.2 Mean count per unique word

First map the `(word, count)` pairs to their count values, then reduce those values to a total.


In [ ]:
from operator import add

totalCount = wordCounts.map(lambda item: item[1]).reduce(add)
average = totalCount / float(uniqueWords)

print(totalCount)
print(round(average, 2))


<a id="6-word-count-helpers"></a>

### 6. Word-Count Helpers

The next cells turn the earlier examples into reusable helper functions. The starter code runs, but it is not yet the finished implementation. Complete the `TODO` items before relying on the Shakespeare counts.

#### 6.1 `wordCount` function

Complete this helper so that it returns one `(word, count)` pair per distinct word.


In [ ]:
def wordCount(wordListRDD):
    # TODO: add the key-based aggregation step so duplicate words are counted together.
    return wordListRDD.map(lambda word: (word, 1))

print(sorted(wordCount(wordsRDD).collect()))


#### 6.2 Capitalisation and punctuation

Real text needs consistent cleaning before counting. Complete `removePunctuation` so that it lowercases text, keeps letters/numbers/spaces, removes punctuation, and strips leading or trailing spaces.


In [ ]:
def removePunctuation(text):
    # TODO: keep only letters, numbers, and spaces before stripping.
    return text.lower().strip()

print(removePunctuation("Hi, you!"))
print(removePunctuation(" No under_score!"))


<a id="7-shakespeare-word-count-case-study"></a>

### 7. Shakespeare Word Count Case Study

For the case study, we use the Complete Works of William Shakespeare text file distributed through the public SIT742 data folder. The source text is from Project Gutenberg.

#### 7.1 Load and clean text lines

The `textFile` call creates one RDD element per line. The `map(removePunctuation)` step applies your cleaning function lazily; an action such as `take` triggers the work.


In [ ]:
fileName = str(shakespeare_path)

shakespeareRDD = (
    sc
    .textFile(fileName, 8)
    .map(removePunctuation)
)

print(shakespeareRDD.zipWithIndex().take(10))


#### 7.2 Words from lines

Use `flatMap` rather than `map` because each line splits into a list of words. `flatMap` flattens those lists into one RDD of word strings.


In [ ]:
shakespeareWordsRDD = shakespeareRDD.flatMap(lambda line: line.split(" "))
shakespeareWordCount = shakespeareWordsRDD.count()

print(shakespeareWordsRDD.top(5))
print(shakespeareWordCount)


#### 7.3 Remove empty elements

Complete this step by filtering out empty strings before calculating the final word count.


In [ ]:
# TODO: filter out empty strings from shakespeareWordsRDD.
shakeWordsRDD = shakespeareWordsRDD
shakeWordCount = shakeWordsRDD.count()

print(shakeWordCount)


#### 7.4 Count the words

Use your completed `wordCount` helper and `takeOrdered` to obtain the 15 most common words. The sort key below orders by the count in descending order.


In [ ]:
top15WordsAndCounts = wordCount(shakeWordsRDD).takeOrdered(15, lambda item: -item[1])
print(top15WordsAndCounts)


<a id="8-student-tasks"></a>

### 8. Student Tasks

Use these tasks after completing the `TODO` cells in the word-count case study. They are designed to extend the same RDD pattern while keeping the notebook as active practice.

1. Add a small stopword list and produce a top-word list after filtering those stopwords.
2. Count how many distinct words begin with the same first letter, then display the five most common first letters.
3. Change the ordering rule for top words so ties are sorted alphabetically after count.
4. Choose one unusual high-frequency token and inspect a few lines where it appears before deciding whether your cleaning rule should change.


In [ ]:
# Student task workspace for M04H.
# Add your word-count extensions below. Keep large text output bounded with take() or takeOrdered().

# Example prompts:
# - Which stopwords should you remove before recounting?
# - How can you map each word to its first letter?
# - What sort key would order by count and then alphabetically?
# - Which bounded action helps inspect a few matching lines?


<a id="9-practical-self-checks"></a>

### 9. Practical Self-Checks

These visible checks are formative prompts. They are designed to help you diagnose your own notebook state after completing the `TODO` items.


In [ ]:
small_counts = dict(wordCount(wordsRDD).collect())
assert small_counts.get("cat") == 2
assert small_counts.get("rat") == 2
assert small_counts.get("elephant") == 1
assert uniqueWords == 3
assert round(average, 2) == 1.67

print("Small RDD checks passed.")


In [ ]:
assert shakeWordCount > 800000
assert len(top15WordsAndCounts) == 15
assert top15WordsAndCounts[0][1] >= top15WordsAndCounts[-1][1]
assert {"the", "and"}.issubset({word for word, count in top15WordsAndCounts[:5]})

print("Shakespeare word-count checks passed.")


<a id="10-reflection-and-references"></a>

### 10. Reflection and References

Reflect on the following questions:

1. Why is `reduceByKey()` usually preferable to `groupByKey()` for word count?
2. What changes if punctuation and capitalisation are not handled consistently?
3. Why should large RDD results be inspected with bounded actions such as `take`, `takeOrdered`, or aggregated counts rather than broad `collect` calls?
4. What extra cleaning step would you add if you wanted content words rather than common stopwords?

References:

- Apache Spark documentation: [RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
- Apache Spark documentation: [PySpark RDD API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.html)
- Project Gutenberg: [The Complete Works of William Shakespeare](https://www.gutenberg.org/ebooks/100)
